# Tutorial: Build And Inspect A Repricing Benchmark Release

This notebook walks through one frozen repricing benchmark release end to end.

The benchmark artifact is intentionally simple:
- `repricing_benchmark.examples`: the frozen manifest of prediction timestamps
- `repricing_benchmark.market_timeseries`: normalized market-level probability histories

For most modeling workflows, the important outputs are still the four core objects:
- `train_df`
- `train_target`
- `test_df`
- `test_target`


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from polymarket_research.benchmarks import RepricingBenchmark, RepricingBenchmarkConfig
from polymarket_research.data.canonical import CanonicalDataset
from polymarket_research.utils.filesystem import setup_root

REPO_ROOT = setup_root()
DATA_SOURCE = 'polymarket'  # or 'kalshi'
ARTEFACT_ROOT = REPO_ROOT / 'frozen_notebooks' / 'running_artefacts' / DATA_SOURCE
CANONICAL_CACHE_DIR = ARTEFACT_ROOT / 'canonical_dataset'
RELEASE_HORIZON_HOURS = 24
RELEASE_NAME = f'{DATA_SOURCE}-repricing-{RELEASE_HORIZON_HOURS}h'
BENCHMARK_CACHE_DIR = ARTEFACT_ROOT / f'repricing_benchmark_{RELEASE_HORIZON_HOURS}h'
MARKET_LIMIT = None
MARKET_ORDER = None
SHOW_PROGRESS = True
USE_BENCHMARK_CACHE = True
BENCHMARK_CONFIG = RepricingBenchmarkConfig(
    future_horizon_hours=RELEASE_HORIZON_HOURS,
    lookback_hours=24,
    sample_every_hours=12,
    move_threshold=0.15,
    attach_external_shocks=True,
    split_on='timestamp_utc',
    train_fraction=0.8,
    show_progress=SHOW_PROGRESS,
)

pd.set_option('display.max_colwidth', None)


## Step 1: Build Or Load One Release

Load the canonical dataset, then either reuse a cached repricing release or build it from scratch.


In [ ]:
canonical = CanonicalDataset.from_parquet(CANONICAL_CACHE_DIR)
print("Loaded canonical from parquet cache:", CANONICAL_CACHE_DIR)


In [ ]:
if USE_BENCHMARK_CACHE and (BENCHMARK_CACHE_DIR / "examples.parquet").exists() and MARKET_LIMIT is None and MARKET_ORDER is None:
    repricing_benchmark = RepricingBenchmark.load(BENCHMARK_CACHE_DIR, canonical=canonical)
    print(f"Loaded {RELEASE_NAME} from parquet cache:", BENCHMARK_CACHE_DIR)
else:
    repricing_benchmark = RepricingBenchmark.build(canonical, config=BENCHMARK_CONFIG)
    if MARKET_LIMIT is None and MARKET_ORDER is None:
        repricing_benchmark.save(BENCHMARK_CACHE_DIR)
    print(f"Built {RELEASE_NAME} from canonical")


In [ ]:
display(canonical.summary())
manifest = repricing_benchmark.manifest()
manifest_frame = pd.DataFrame([
    {"field": key, "value": repr(value) if isinstance(value, (list, dict, tuple)) else value}
    for key, value in manifest.items()
])
display(manifest_frame)

display(pd.DataFrame([
    {
        "release_name": repricing_benchmark.release_name,
        "market_timeseries_rows": len(repricing_benchmark.market_timeseries),
        "market_timeseries_columns": list(repricing_benchmark.market_timeseries.columns),
    }
]))


## Step 2: Materialize The Four Main Objects

These are usually the first tables you want for training and evaluation.


In [ ]:
train_df = repricing_benchmark.split_examples("train")
train_target = repricing_benchmark.targets("train")
test_df = repricing_benchmark.split_examples("test")
test_target = repricing_benchmark.targets("test")


In [ ]:
display(train_df.head())
display(train_target.head())
display(test_df.head())
display(test_target.head())

preview_market_ids = train_df.head(3)["market_id"]
train_timeseries_preview = repricing_benchmark.market_timeseries.loc[
    lambda df: df["market_id"].isin(preview_market_ids)
].reset_index(drop=True)
display(train_timeseries_preview.head(20))


## Step 3: Inspect One Held-Out Snapshot

Repricing has multiple benchmark rows per market, so the natural key is `(market_id, timestamp_utc)`.


In [ ]:
snapshot = test_df.sample().iloc[0]
market_id = snapshot["market_id"]
timestamp_utc = snapshot["timestamp_utc"]
example = repricing_benchmark.resolve_market_snapshot(market_id, timestamp_utc)
observation = repricing_benchmark.history_until(market_id, timestamp_utc)

display(example.to_frame(name="value"))
display(observation.tail(20))


In [ ]:
observation.set_index("timestamp_utc")["yes_probability"].plot(figsize=(10, 4), color="#1971C2", linewidth=2)
plt.title(f"Repricing observation prefix for {market_id}")
plt.xlabel("timestamp_utc")
plt.ylabel("yes_probability")
plt.show()


## Step 4: Build A Lightweight Reference View

The frozen artifact stays lean. If you need engineered repricing features, derive the reference view explicitly.


In [ ]:
repricing_view = repricing_benchmark.build_reference_view()

display(repricing_view.head())

category_summary = (
    repricing_view.groupby('research_category', dropna=False)
    .agg(
        rows=('market_id', 'size'),
        positive_rate=('target', 'mean'),
        future_move_mean=('future_move', 'mean'),
    )
    .reset_index()
    .sort_values('rows', ascending=False, kind='stable')
)
display(category_summary.head(20))


## Step 5: Score A Simple Baseline

Predictions for repricing are keyed by `market_id` and `timestamp_utc`.


In [ ]:
test_predictions = test_df.loc[:, ["market_id", "timestamp_utc"]].copy()
test_predictions["pred_prob"] = 0.5

evaluation = repricing_benchmark.evaluate(test_predictions, split="test")
display(evaluation["overall"])
display(evaluation["by_horizon"])


## Step 6: Sanity-Check The Split

The main checks are split sizes, label balance, and how many prediction timestamps come from each market.


In [ ]:
split_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "examples": len(train_df),
            "targets": len(train_target),
            "positive_rate": train_target["label"].mean(),
            "market_count": train_df["market_id"].nunique(),
        },
        {
            "split": "test",
            "examples": len(test_df),
            "targets": len(test_target),
            "positive_rate": test_target["label"].mean(),
            "market_count": test_df["market_id"].nunique(),
        },
    ]
)
display(split_summary)

snapshot_summary = (
    repricing_benchmark.examples.groupby(["split", "market_id"], dropna=False)
    .size()
    .rename("repricing_rows")
    .reset_index()
)
display(snapshot_summary.head(20))


## Step 7: Plot A Small Audit View

These plots use the frozen manifest plus the explicit feature view you already derived.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

(
    repricing_benchmark.examples.groupby("split")["market_id"]
    .size()
    .plot(kind="bar", ax=axes[0], color=["#495057", "#1971C2"])
)
axes[0].set_title("Repricing examples by split")
axes[0].set_xlabel("split")
axes[0].set_ylabel("rows")

repricing_view.groupby("confidence_slice")["target"].mean().plot(
    kind="bar", ax=axes[1], color="#339AF0"
)
axes[1].set_title("Positive rate by confidence slice")
axes[1].set_xlabel("confidence slice")
axes[1].set_ylabel("positive rate")

plt.tight_layout()
plt.show()
